# SAE-Gated RAG: full run on SageMaker (one L4)
 
# v2

Runs everything left that needs a GPU, then the CPU probe work, in one go. **Run All.** The only prompt is a GitHub token at the start: the repo is private, and the token also pushes the results back to the `aws` branch.

| Step | What | Rough time on `ml.g6.2xlarge` |
|---|---|---|
| 1 | Closed-book, BM25 and gold generations for every split (Gemma 3 4B, bf16) | 2–3 h |
| 2 | Labels (both rules) and the Gate 0 report | seconds |
| 3 | Activations at layers 4/9/13/17/22/29 and 16k SAE codes at 9/17/22/29 | ~30 min |
| 4 | B4 verbalised router | ~15 min |
| 5 | Batch-size-1 latency: controller, closed-book, BM25-RAG, B4 | ~20 min |
| 6 | Hand-check sheet, B1–B3 and B8 | minutes |
| 7 | Diagnostic for the 1,000 vs 2,000 label SAE result | ~1 h |
| 8 | Three-way probe grid (main label rule), extra dense layers, other rule at full labels | 3–4 h |
| 9 | Two-way routers (for H3), scoring, routed latency (for H4) | minutes |

Every step writes a marker in `runs/.done/`, so if anything fails or the space stops, **run all again and it resumes**. The full log is in `logs/run_all.log`.

**Before you start:** the space needs at least 30 GB of free storage (set it to 50 GB in Space Settings). Stop the space when it's done, because it bills while idle.

**Results** go to `runs/` and `data/` on the space, and a `results_bundle.tar.gz` at the end. The small result files (reports, scores, probe summaries, latency, diagnostics, hand-check sheet, stats, log) are also copied to `results/sagemaker/` and committed to the `aws` branch, after the GPU part, at the end, and if a step fails. Raw labels, generations, probabilities and features stay out of git; they are in the bundle.

**Not covered here:** BM25 retrieval latency needs the Lucene index, which is on ADA. Measure it there and add it to the RETRIEVE path in the latency table.

## 1. Settings

In [ ]:
PRIMARY = "miss_as_retrieve"   # label rule for the main grid, baselines and scoring; the other rule gets full-label probes only
LAYER = 17                     # probe layer used for latency, the anomaly diagnostic and the two-way routers
SEEDS = "0 1 2"                # seeds for each label budget
WORKDIR = "~/lma-sae-rag-proj"
REPO = "github.com/bropal404/lma-sae-rag-proj.git"
BRANCH = "aws"                 # branch with this notebook; results are pushed here
COMMIT_RESULTS = True          # commit and push the small result files (needs a token with write access)
GIT_NAME, GIT_EMAIL = "Gopal Kataria", "gopal.kataria@research.iiit.ac.in"

## 2. Check the machine, get the code

In [ ]:
import getpass, os, shutil, subprocess, sys

WORKDIR = os.path.expanduser(WORKDIR)
def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, executable="/bin/bash")

sh("nvidia-smi")
free_gb = shutil.disk_usage(os.path.expanduser("~")).free / 1e9
print(f"free disk in home: {free_gb:.0f} GB")
assert free_gb > 30, "Need about 30 GB free: stop the space, raise its storage to 50 GB in Space Settings, start again."

token = ""
if COMMIT_RESULTS or not os.path.exists(WORKDIR):
    token = getpass.getpass("GitHub token (fine-grained, this repo, Contents: read and write): ").strip()
os.environ["GH_TOKEN"] = token          # kept in memory only; git reads it through nb_tools/askpass.sh
if not os.path.exists(WORKDIR):
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, WORKDIR], check=True)
    subprocess.run(["git", "-C", WORKDIR, "remote", "set-url", "origin", f"https://{REPO}"], check=True)  # keep the token out of .git/config
else:
    print("repo already cloned:", WORKDIR)
os.chdir(WORKDIR)
os.makedirs("nb_tools", exist_ok=True)
os.makedirs("logs", exist_ok=True)
with open("nb_tools/askpass.sh", "w") as f:
    f.write('#!/bin/sh\ncase "$1" in Username*) echo x-access-token ;; *) echo "$GH_TOKEN" ;; esac\n')
os.chmod("nb_tools/askpass.sh", 0o700)
sh(f'git config user.name "{GIT_NAME}" && git config user.email "{GIT_EMAIL}"')
sh("git log -1 --oneline")

## 3. Python environment

The repo's lockfile pins a CUDA 13 build of torch, which needs a very new driver. This installs a torch build that matches the driver on this machine, then the project with the same `transformers` and `accelerate` versions the team used. It uses the Hugging Face backend, not vLLM.

In [ ]:
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]
if shutil.which("uv") is None:
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh")

driver = int(subprocess.check_output(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]).decode().split(".")[0])
print("NVIDIA driver", driver)
PY = ".venv/bin/python"
if not os.path.exists(PY):
    sh("uv venv --python 3.12 .venv")

def cuda_ok():
    return subprocess.run([PY, "-c", "import torch; assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()"]).returncode == 0

if not cuda_ok():
    for flavour in (["cu130", "cu128"] if driver >= 580 else ["cu128", "cu126"]):
        r = subprocess.run(f"uv pip install --python {PY} --reinstall-package torch torch --index-url https://download.pytorch.org/whl/{flavour}",
                           shell=True, executable="/bin/bash")
        if r.returncode == 0 and cuda_ok():
            print("torch build:", flavour)
            break
    else:
        raise RuntimeError("No torch build sees the GPU; check nvidia-smi output above.")
sh(f'uv pip install --python {PY} -e ".[features]" "transformers==5.19.0" "accelerate==1.15.0"')
sh(f'{PY} -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))"')

## 4. Data, config and model weights

Questions and BM25 / gold passages come from Person 1's handoff in the repo. The model is `unsloth/gemma-3-4b-it`, an ungated mirror with the same weight hashes as `google/gemma-3-4b-it`, so no Hugging Face login is needed.

In [ ]:
os.environ["HF_HOME"] = os.path.expanduser("~/hf")
if not os.path.exists("data/questions/pilot.jsonl"):
    os.makedirs("data", exist_ok=True)
    sh("tar xzf artifacts/retrieval/person1-full-wikipedia-handoff.tar.gz -C data questions retrieval")
sh("ls data/questions data/retrieval/bm25 | head -30")

with open("configs/sagemaker.yaml", "w") as f:
    f.write('''# One L4 (bf16) on SageMaker, Hugging Face backend.
extends: default.yaml
model:
  generator: unsloth/gemma-3-4b-it
generation:
  backend: hf
  batch_size: 32
''')
sh(f"{PY} -m saerag.prefetch --config configs/sagemaker.yaml --widths 16k")

## 5. Helper scripts

These use the project's own modules and are written into `nb_tools/` (not part of the repo).

In [ ]:
%%writefile nb_tools/latency_measure.py
"""Batch-size-1 timings on this GPU for questions from nq_test and triviaqa_test.

Per question: controller (forward pass + SAE encode + probe), closed-book generation,
BM25-RAG generation (generation only; BM25 search time is measured on ADA) and the B4
verbalised router. The probe here has a fixed C: its prediction cost does not depend on C.
Writes runs/latency/stages.jsonl and runs/latency/stages_summary.csv.
"""
import argparse, json, time

import numpy as np, pandas as pd, torch
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MaxAbsScaler

from saerag.config import load_config, path
from saerag.features.sae import JumpReLUSAE, sae_id
from saerag.hf import chat_prompt, load_model
from saerag.probes.train import load_block
from saerag.prompts import closed_book_messages, rag_messages, verbalised_router_messages
from saerag.schema import Question, Retrieval, read_jsonl

ap = argparse.ArgumentParser()
ap.add_argument("--config")
ap.add_argument("--layer", type=int, default=17)
ap.add_argument("--variant", default="miss_as_retrieve")
ap.add_argument("--files", nargs="+", default=["nq_test", "triviaqa_test"])
ap.add_argument("--n", type=int, default=100, help="questions per file")
args = ap.parse_args()
cfg = load_config(args.config)
m, g, L = cfg["model"], cfg["generation"], args.layer

name = f"sae16k_last@L{L}"
_, y, _, X = load_block(cfg, ["nq_train", "triviaqa_train"], args.variant, [L], ["sae16k_last"])
probe = make_pipeline(MaxAbsScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)).fit(X[name], y)

tok, model, dev = load_model(m["generator"], g.get("dtype", "auto"))
sae = JumpReLUSAE.load(m["sae_repo"], sae_id(m["sae_site"], L, "16k", m["sae_l0"]), dev, decoder=False)
k = cfg["retrieval"]["top_k"]

def enc(msgs):
    return tok(chat_prompt(tok, msgs), return_tensors="pt", add_special_tokens=False).to(dev)

def timed(fn):
    torch.cuda.synchronize(); t = time.perf_counter(); fn(); torch.cuda.synchronize()
    return (time.perf_counter() - t) * 1000

def controller(q):
    with torch.no_grad():
        h = model(**enc(closed_book_messages(q.question)), output_hidden_states=True).hidden_states[L + 1][:, -1]
        codes = sae.encode(h.float()).cpu().numpy()
    probe.predict_proba(codes)

def generate(msgs, n_tokens):
    with torch.no_grad():
        model.generate(**enc(msgs), max_new_tokens=n_tokens, do_sample=False)

rows = []
for f in args.files:
    qs = list(read_jsonl(path(cfg, "questions", f"{f}.jsonl"), Question))[: args.n]
    ret = {r.qid: r for r in read_jsonl(path(cfg, "retrieval", "bm25", f"{f}.jsonl"), Retrieval)}
    for q in qs[:3]:                                   # warm-up
        controller(q); generate(closed_book_messages(q.question), 4)
    for q in qs:
        rows.append({"qid": q.qid, "file": f,
                     "controller_ms": timed(lambda: controller(q)),
                     "closed_ms": timed(lambda: generate(closed_book_messages(q.question), g["max_new_tokens"])),
                     "rag_ms": timed(lambda: generate(rag_messages(q.question, ret[q.qid].passages[:k]), g["max_new_tokens"])),
                     "b4_ms": timed(lambda: generate(verbalised_router_messages(q.question), 8))})
    print(f"{f}: timed {len(qs)} questions", flush=True)

out = path(cfg, "results", "latency")
out.mkdir(parents=True, exist_ok=True)
with open(out / "stages.jsonl", "w") as fh:
    fh.writelines(json.dumps(r) + "\n" for r in rows)
df = pd.DataFrame(rows)
cols = ["controller_ms", "closed_ms", "rag_ms", "b4_ms"]
summary = pd.DataFrame({"p50": df[cols].median(), "p95": df[cols].quantile(0.95), "mean": df[cols].mean()}).round(1)
summary.to_csv(out / "stages_summary.csv")
print(summary.to_string())

In [ ]:
%%writefile nb_tools/anomaly.py
"""Why did the SAE probe score lower on 2,000 labels than on 1,000 (mid-eval report)?

Refits SAE and dense probes at one layer for several label budgets and records, per fit,
the C picked by the inner CV, the inner-CV score, iterations used (5,000 = did not converge),
non-zero weights and test scores on nq_test + triviaqa_test. Runs on the proxy label
("closed": closed-book correct or not) and on the main label rule.
Writes runs/diagnostics/anomaly.csv.
"""
import argparse

import numpy as np, pandas as pd

from saerag.config import load_config, path
from saerag.probes.train import load_block, make_model, score, subsample

ap = argparse.ArgumentParser()
ap.add_argument("--config")
ap.add_argument("--layer", type=int, default=17)
ap.add_argument("--labels", nargs="+", default=["closed", "miss_as_retrieve"])
ap.add_argument("--budgets", nargs="+", type=int, default=[1000, 2000, 4000])
ap.add_argument("--seeds", nargs="+", type=int, default=[0, 1, 2])
args = ap.parse_args()
cfg = load_config(args.config)
L, kinds = args.layer, ["resid_last", "sae16k_last"]

rows = []
for labels in args.labels:
    _, y_str, _, X = load_block(cfg, ["nq_train", "triviaqa_train"], labels, [L], kinds)
    _, y2_str, _, X2 = load_block(cfg, ["nq_test", "triviaqa_test"], labels, [L], kinds)
    classes = sorted(set(y_str))
    y = np.array([classes.index(c) for c in y_str])
    y2 = np.array([classes.index(c) for c in y2_str])
    idx = np.arange(len(y))
    for k in kinds:
        name = f"{k}@L{L}"
        for budget in [*args.budgets, None]:
            for seed in (args.seeds if budget else args.seeds[:1]):
                tr = subsample(idx, y, budget, seed)
                fit = make_model(name, "l2", False, seed).fit(X[name][tr], y[tr])
                lr = fit.best_estimator_[-1]
                rows.append({"labels": labels, "model": name, "budget": len(tr), "seed": seed,
                             "C": fit.best_params_["logisticregression__C"],
                             "inner_cv_f1": fit.best_score_, "n_iter": int(np.max(lr.n_iter_)),
                             "nonzero_weights": int((lr.coef_ != 0).sum()),
                             **score(fit.predict_proba(X2[name]), y2, len(classes))})
                r = rows[-1]
                print(f"{labels} {name} n={r['budget']} seed={seed} C={r['C']} iters={r['n_iter']} "
                      f"test F1={r['macro_f1']:.3f}", flush=True)

df = pd.DataFrame(rows)
out = path(cfg, "results", "diagnostics")
out.mkdir(parents=True, exist_ok=True)
df.to_csv(out / "anomaly.csv", index=False)
print(df.groupby(["labels", "model", "budget"])[["macro_f1", "inner_cv_f1", "n_iter"]].mean().round(3).to_string())
print(df.groupby(["labels", "model", "budget"]).C.agg(lambda c: sorted(set(c))).to_string())

In [ ]:
%%writefile nb_tools/twoway.py
"""Two-way routers for H3: each three-way probe with ABSTAIN folded into RETRIEVE."""
import argparse

from saerag.config import load_config, path
from saerag.schema import read_jsonl, write_jsonl

ap = argparse.ArgumentParser()
ap.add_argument("--config")
ap.add_argument("--variant", required=True)
ap.add_argument("--layers", nargs="+", type=int, required=True)
ap.add_argument("--files", nargs="+", required=True)
args = ap.parse_args()
cfg = load_config(args.config)

for L in args.layers:
    for kind in ("resid_last", "sae16k_last"):
        pid = f"{cfg['model']['tag']}__{args.variant}__{kind}@L{L}__l2"
        for f in args.files:
            src = path(cfg, "results", "routers", pid, f"probs_{f}.jsonl")
            if not src.exists():
                continue
            rows = [{"qid": r["qid"], "p": {"ANSWER": r["p"].get("ANSWER", 0.0),
                                            "RETRIEVE": r["p"].get("RETRIEVE", 0.0) + r["p"].get("ABSTAIN", 0.0),
                                            "ABSTAIN": 0.0}} for r in read_jsonl(src)]
            write_jsonl(path(cfg, "results", "routers", f"{pid}__2way", f"probs_{f}.jsonl"), rows)
        print("wrote", f"{pid}__2way")

In [ ]:
%%writefile nb_tools/latency_route.py
"""Per-router latency on the timed questions (H4), with argmax routing.

ANSWER costs a closed-book generation, RETRIEVE a BM25-RAG generation (plus BM25 search,
measured separately on ADA), ABSTAIN nothing. Probes add the controller time, B4 its own
generation, and B8 nothing measurable. Writes runs/latency/routed.csv.
"""
import argparse, json

import numpy as np, pandas as pd

from saerag.config import load_config, path
from saerag.eval.metrics import argmax_actions
from saerag.schema import read_jsonl

ap = argparse.ArgumentParser()
ap.add_argument("--config")
ap.add_argument("--variant", required=True)
ap.add_argument("--layer", type=int, default=17)
args = ap.parse_args()
cfg = load_config(args.config)
tag = cfg["model"]["tag"]

stages = pd.read_json(path(cfg, "results", "latency", "stages.jsonl"), lines=True).set_index("qid")
probes = [f"{tag}__{args.variant}__{k}@L{args.layer}__l2{s}" for k in ("resid_last", "sae16k_last") for s in ("", "__2way")]
routers = ["b1_always_retrieve", "b2_never_retrieve", "b3_oracle", "b4_verbalised", "b8_tfidf_lr", *probes]

rows = []
for rid in routers:
    probs = {}
    for f in stages.file.unique():
        p = path(cfg, "results", "routers", rid, f"probs_{f}.jsonl")
        if p.exists():
            probs |= {r["qid"]: r["p"] for r in read_jsonl(p) if r["qid"] in stages.index}
    if not probs:
        print("no probabilities for", rid); continue
    acts = argmax_actions(probs)
    ctrl = "controller_ms" if rid in probes else ("b4_ms" if rid == "b4_verbalised" else None)
    lat = []
    for q, a in acts.items():
        s = stages.loc[q]
        lat.append((s[ctrl] if ctrl else 0.0) + {"ANSWER": s.closed_ms, "RETRIEVE": s.rag_ms, "ABSTAIN": 0.0}[a])
    rows.append({"router": rid, "n": len(lat), "retrieval_calls": np.mean([a == "RETRIEVE" for a in acts.values()]),
                 "p50_ms": np.median(lat), "p95_ms": np.percentile(lat, 95), "mean_ms": np.mean(lat)})
df = pd.DataFrame(rows).round(3)
df.to_csv(path(cfg, "results", "latency", "routed.csv"), index=False)
print(df.to_string(index=False))

In [ ]:
%%writefile nb_tools/run_all.sh
#!/bin/bash
# Everything in order. Each finished step leaves runs/.done/<step>, so rerunning resumes.
set -euo pipefail
cd "$(dirname "$0")/.."
PY=.venv/bin/python
C="--config configs/sagemaker.yaml"
PRIMARY=${PRIMARY:-miss_as_retrieve}
if [ "$PRIMARY" = default ]; then SECONDARY=miss_as_retrieve; else SECONDARY=default; fi
LAYER=${LAYER:-17}
SEEDS=${SEEDS:-"0 1 2"}
ALL="pilot nq_train nq_val nq_test triviaqa_train triviaqa_val triviaqa_test popqa_test hotpotqa_test selfaware_test"
TRAIN="nq_train triviaqa_train"
EVAL="nq_val triviaqa_val nq_test triviaqa_test popqa_test hotpotqa_test selfaware_test"
TEST="nq_test triviaqa_test popqa_test hotpotqa_test selfaware_test"
LAYERS="9 17 22 29"
TAG=gemma3-4b-it
mkdir -p runs/.done

step() {
  local name=$1; shift
  if [ -f "runs/.done/$name" ]; then echo "== skip $name (done)"; return; fi
  echo "== $(date '+%F %T') start $name"
  local t0=$SECONDS
  "$@"
  touch "runs/.done/$name"
  echo "== $name done in $(( (SECONDS - t0) / 60 )) min"
}
gate0() { $PY -m saerag.generation.report $C --files pilot || echo "Gate 0: a class is under 15% under the default rule (see runs/reports/)"; }
publish() {   # copy the small result files to results/sagemaker/, commit, push to $BRANCH
  [ "${COMMIT_RESULTS:-1}" = 1 ] || return 0
  local out=results/sagemaker
  mkdir -p $out
  ( cd runs && find reports scores latency diagnostics handcheck -type f 2>/dev/null | xargs -r cp --parents -t ../$out ) || true
  ( cd runs && find probes -type f \( -name summary.csv -o -name 'top_features_*.json' \) 2>/dev/null | xargs -r cp --parents -t ../$out ) || true
  ( cd data && find features generations -type f \( -name 'stats_*.json' -o -name '*.stats.json' \) 2>/dev/null | xargs -r cp --parents -t ../$out ) || true
  if [ -f logs/run_all.log ]; then cp logs/run_all.log $out/; fi
  git add $out
  if git diff --cached --quiet; then echo "publish: nothing new"; return 0; fi
  git commit -q -m "SageMaker results: $1" -m "Run on $(nvidia-smi --query-gpu=name --format=csv,noheader | head -1), label rule $PRIMARY, probe layer $LAYER."       -m "Co-Authored-By: Amazon SageMaker <sagemaker@noreply.invalid>"
  echo "publish: committed $1"
  if [ -n "${GH_TOKEN:-}" ]; then
    export GIT_ASKPASS=$PWD/nb_tools/askpass.sh GIT_TERMINAL_PROMPT=0
    { git pull -q --rebase origin "$BRANCH" && git push -q origin HEAD:"$BRANCH" && echo "publish: pushed to $BRANCH"; }       || echo "publish: push failed; the commit is still local"
  fi
}
score_all() {
  local routers="b1_always_retrieve b2_never_retrieve b3_oracle b4_verbalised b8_tfidf_lr"
  for L in $LAYERS; do for k in resid_last sae16k_last; do
    routers="$routers ${TAG}__${PRIMARY}__${k}@L${L}__l2 ${TAG}__${PRIMARY}__${k}@L${L}__l2__2way"
  done; done
  for f in $TEST; do
    local val=nq_val; [ "$f" = triviaqa_test ] && val=triviaqa_val
    $PY -m saerag.eval.score $C --routers $routers --files $f --variant $PRIMARY --val-file $val
  done
}

BRANCH=${BRANCH:-aws}
trap 'publish "partial, a step failed" || true' ERR
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
echo "main label rule: $PRIMARY   probe layer: $LAYER   seeds: $SEEDS"

# ---- GPU ----
step generate   $PY -m saerag.generation.run $C --condition closed bm25 gold --files $ALL --backend hf
step labels     $PY -m saerag.generation.make_labels $C --files $ALL
step gate0      gate0
step features   $PY -m saerag.features.extract $C --files $ALL --widths 16k
step b4         $PY -m saerag.baselines.run $C --routers b4 --files $EVAL --backend hf --variant $PRIMARY
step latency    $PY nb_tools/latency_measure.py $C --layer $LAYER --variant $PRIMARY
publish "GPU stage (generations, labels, features, B4, latency)" || true

# ---- CPU ----
step handcheck  $PY -m saerag.eval.handcheck $C sample --files nq_test triviaqa_test popqa_test hotpotqa_test --n 200 --variant $PRIMARY
step baselines  $PY -m saerag.baselines.run $C --routers b1 b2 b3 b8 --files $EVAL --variant $PRIMARY
step anomaly    $PY nb_tools/anomaly.py $C --layer $LAYER --labels closed $PRIMARY --seeds $SEEDS
step grid_main  $PY -m saerag.probes.train $C --train $TRAIN --eval $EVAL --labels $PRIMARY \
                    --layers $LAYERS --kinds resid_last sae16k_last --budgets 50 100 200 500 1000 2000 5000 --seeds $SEEDS
step dense_extra $PY -m saerag.probes.train $C --train $TRAIN --eval $EVAL --labels $PRIMARY \
                    --layers 4 13 --kinds resid_last --name extra_layers
step grid_other $PY -m saerag.probes.train $C --train $TRAIN --eval $EVAL --labels $SECONDARY \
                    --layers $LAYERS --kinds resid_last sae16k_last
step twoway     $PY nb_tools/twoway.py $C --variant $PRIMARY --layers $LAYERS --files $EVAL
step score      score_all
step routed_latency $PY nb_tools/latency_route.py $C --variant $PRIMARY --layer $LAYER

tar czf results_bundle.tar.gz runs data/labels data/generations data/features/$TAG/stats_*.json logs
publish "full run" || true
echo "== all done: results_bundle.tar.gz"

## 6. Run everything

Output streams here and goes to `logs/run_all.log`. If the browser disconnects, the kernel keeps running; open the log to follow it. If the cell fails, fix the cause and run it again: finished steps are skipped.

In [ ]:
import subprocess
env = {**os.environ, "PRIMARY": PRIMARY, "LAYER": str(LAYER), "SEEDS": SEEDS, "BRANCH": BRANCH,
       "COMMIT_RESULTS": "1" if COMMIT_RESULTS else "0",
       "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false", "TQDM_MININTERVAL": "60"}
with open("logs/run_all.log", "a") as log:
    p = subprocess.Popen(["bash", "nb_tools/run_all.sh"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, env=env, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
        log.write(line); log.flush()
    rc = p.wait()
assert rc == 0, f"run_all.sh failed (exit {rc}); see logs/run_all.log, then run this cell again to resume"

## 7. Results

In [ ]:
import glob
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
TAG = "gemma3-4b-it"

print("==== Gate 0 (pilot) ====")
for f in glob.glob(f"runs/reports/{TAG}/pilot.md"):
    print(open(f).read())

print("==== Hand-check sheet (Gate 1): fill in, then run handcheck score ====")
print(glob.glob("runs/handcheck/*.csv"))

In [ ]:
print("==== 1,000 vs 2,000 label diagnostic ====")
a = pd.read_csv("runs/diagnostics/anomaly.csv")
display(a.groupby(["labels", "model", "budget"])[["macro_f1", "auroc", "inner_cv_f1", "n_iter"]].mean().round(3))
display(a.groupby(["labels", "model", "budget"]).C.agg(lambda c: sorted(set(c))))

In [ ]:
print("==== Probe grid: mean macro-F1 by label budget (main rule) ====")
for f in sorted(glob.glob(f"runs/probes/{TAG}/{PRIMARY}/*/summary.csv")):
    s = pd.read_csv(f)
    print(f)
    display(s[s["eval"].isin(["nq_test", "triviaqa_test", "popqa_test", "hotpotqa_test"])]
            .pivot_table(index=["eval", "model"], columns="budget", values="macro_f1", aggfunc="mean").round(3))

In [ ]:
print("==== Routers (threshold tuned on val), all questions / hole / non-hole ====")
cols = ["router", "subset", "n", "accuracy", "hallucination_rate", "retrieval_call_rate",
        "abstention_recall", "false_refusal_rate", "aurc", "macro_f1"]
for f in sorted(glob.glob(f"runs/scores/{PRIMARY}/*.csv")):
    s = pd.read_csv(f)
    print(f)
    display(s[s.subset.isin(["all", "hole", "no_hole"])][[c for c in cols if c in s]].round(3))

In [ ]:
print("==== Latency at batch size 1 (ms) ====")
display(pd.read_csv("runs/latency/stages_summary.csv", index_col=0))
display(pd.read_csv("runs/latency/routed.csv"))
print("Download results_bundle.tar.gz from the file browser (right-click, Download). Then stop the space.")